In [ ]:
!unzip "/content/drive/MyDrive/Liver cyst seg.zip"

In [ ]:
%cd /content

In [ ]:
!pip install ultralytics


In [ ]:
import os

from ultralytics import YOLO


model = YOLO('yolov8n-seg.pt')  # load a pretrained model (recommended for training)

model.train(data='/content/data.yaml', epochs=50, imgsz=640)



In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Paths
model_path = '/content/runs/segment/train/weights/best.pt'
image_path = '/content/test/images/DataSet12_Abscess_PortalVenous_jpg.rf.e49d1113af33a8d7b3dcb39da932a6d1.jpg'

# Load image
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # For display
H, W, _ = img.shape

# Load model
model = YOLO(model_path)

# Run inference
results = model(img)

# Display original
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis("off")

for result_idx, result in enumerate(results):
    if result.masks is not None:
        for j, (mask_tensor, box) in enumerate(zip(result.masks.data, result.boxes)):
            # Get mask
            mask = mask_tensor.cpu().numpy() * 255
            mask = cv2.resize(mask, (W, H)).astype(np.uint8)

            # Display mask
            plt.subplot(1, 3, 2)
            plt.imshow(mask, cmap='gray')
            plt.title("Mask")
            plt.axis("off")

            # Create overlay
            overlay = img_rgb.copy()
            color = (0, 255, 0)  # green for overlay
            overlay[mask > 0] = (0.4 * overlay[mask > 0] + 0.6 * np.array(color)).astype(np.uint8)

            # Get class name
            cls_id = int(box.cls.cpu().numpy())
            cls_name = model.names[cls_id]

            # Draw bounding box
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (255, 0, 0), 2)

            # Put class name
            cv2.putText(overlay, cls_name, (x1, max(y1 - 10, 0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

            # Display overlay
            plt.subplot(1, 3, 3)
            plt.imshow(overlay)
            plt.title(f"Overlay - {cls_name}")
            plt.axis("off")

plt.tight_layout()
plt.show()
